# 실습 01 · 에이전트 루프 직접 만들기 — 모델은 호출을 생성하고, 루프가 실행한다

**직접 해 보는 것** : 도구 없이 한 번 묻는 것(단발 호출)에서 출발해, 같은 모델에 도구를 주고 **루프를 직접 짜서** 에이전트를 만든다. 모델이 돌려주는 것이 무엇인지 날것으로 확인하고, 루프가 쌓아 가는 메시지 열(컨텍스트)과 궤적을 그림으로 본다. 마지막으로 같은 질문 8개를 단발과 루프로 각각 풀어 성공률 · 토큰 · 지연을 측정한다.

**강의의 어느 개념과 이어지는가**
- 에이전트 = **LLM(정책) + 도구 + 루프**, 제어 흐름을 모델이 쥠
- 「마법이 아니라 껍데기」 : 출력의 일부가 자연어가 아니라 **도구 호출**일 뿐이고, 그 호출을 실행해 결과를 컨텍스트에 되먹이는 루프가 곧 에이전트
- 루프의 의사코드 : `상태 = [system, user]` → `행동 = LLM(상태)` → `관찰 = 실행(행동)` → `상태 += [행동, 관찰]`

**문제** : 가상 쇼핑몰의 환불 상담(주문 DB · 환불 정책은 노트북이 만든 합성 데이터). 「주문 A1003은 환불할 수 있나요? 가능하면 금액은?」 — 모델은 이 가게의 DB를 모르므로 도구가 필요하다. 정답은 정책을 코드로 옮겨 계산한다(사람이 채점하지 않음).

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 · 문제 확인 |
| 2 | 단발 호출 — 도구 없이 한 번 |
| 3 | 도구 한 개의 정체 — 이름 · 설명 · 스키마 · 함수 |
| 4 | 도구 목록을 주고 한 번 — 돌아온 「호출 문장」 |
| 5 | 호출을 우리가 실행해 되먹이기 — 손으로 한 바퀴 |
| 6 | 루프를 직접 짜기(의사코드 → 코드) |
| 7 | 루프를 돌리고 메시지 열 들여다보기 |
| 8 | 완성형 `run_agent` — 궤적 그림 · 컨텍스트 누적 |
| 9 | 루프의 안전장치 — 없는 도구 · 끝나지 않는 정책 |
| 10 | 단발 vs 루프 — 같은 질문 8개 측정 |
| 11 | 정리 · 결과 저장 · 확인 질문 |

- 실행 : LM Studio 서버가 켜져 있고 도구 호출이 되는 모델이 로드되어 있어야 한다(없으면 `AGENT_MODE=replay` 로 강사 실행 결과를 재생). 모든 설정은 출력에 찍는다
- 처음 호출한 응답은 `cache/` 에 저장되어 다시 돌릴 때 같은 결과를 쓴다. 다시 측정하려면 `LLM(mode="live")`

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
import sys, json, time; sys.path.insert(0, ".")
from agent_lab import *
from agent_tasks import *

CFG = dict(max_steps=8, max_tokens=400)
llm = LLM()          # 주소 · 모델은 환경변수(LMSTUDIO_URL · AGENT_MODEL)에서 읽는다. 직접 주려면 LLM(model="...", url="...")
orders, cust, QS = make_questions_D(0)
QS1 = [q for q in QS if q["유형"] == "주문번호"]              # 주문번호가 주어진 질문 8개
Q = next(q for q in QS1 if q["expect"][0] == "가능")          # 이번에 따라가 볼 질문 하나
TOOLS = tools_D(orders, cust)
print(llm, "| 설정 :", CFG)
print("질문 :", Q["q"])
print("정답(코드가 정책에서 계산) :", Q["expect"], "| 주문 :", orders[Q["oid"]])

### 셀 2 · 단발 호출 — 도구 없이 한 번 묻는다
- 프롬프트 한 번 → 답 한 번. 경로를 사람이 고정한 가장 단순한 호출
- 모델이 아는 것은 **학습한 지식**뿐이다. 이 가상 가게의 주문 DB와 환불 정책은 모른다 → 지어내거나 모른다고 답한다
- 채점은 정답 금액이 답에 들어 있는지 규칙으로 본다
- 모델에 따라 도구를 주지 않았는데도 도구 호출처럼 보이는 글(`<tool_call>` 같은 표기)을 답으로 내기도 한다. 이것도 답이 아니므로 실패로 채점한다

In [ ]:
# 셀 2 · 단발 호출
msgs = [{"role": "system", "content": SYSTEM_D}, {"role": "user", "content": Q["q"]}]
r = llm.chat(msgs, max_tokens=CFG["max_tokens"])
print("모델의 답 :", r.content[:400])
print("채점 :", score_D(Q, r.content))
print(f"토큰 {r.tokens} · 지연 {r.latency:.1f}초" + (" (캐시 — 처음 호출했을 때의 값)" if r.cached else ""))

### 셀 3 · 도구 한 개의 정체
- 도구 = **이름 + 설명 + 인자 스키마**(모델에게 보이는 글) + **실제 함수**(모델은 모름)
- 모델은 스키마를 읽고 「이 도구를 이런 인자로 부르겠다」는 문장을 만들 뿐이다. 함수를 실행하는 것은 우리 코드다

In [ ]:
# 셀 3 · 도구 한 개의 정체
print("모델에게 보이는 것(스키마) :")
print(json.dumps(TOOLS[0].schema(), ensure_ascii=False, indent=1))
print("\n모델이 모르는 것(실제 함수) → 우리가 부르면 :", TOOLS[0].fn(order_id=Q["oid"])[:150])
print("\n도구 이름 :", [t.name for t in TOOLS])

### 셀 4 · 도구 목록을 주고 한 번 묻는다
- 같은 질문에 `tools=` 로 도구 목록을 함께 준다
- 돌아오는 것은 최종 답이 아니라 **도구 호출**(`tool_calls`)일 수 있다. 인자(`arguments`)가 **문자열화된 JSON** 으로 오는 점을 본다
- 이 셀에서도 어떤 함수도 실행되지 않는다

In [ ]:
# 셀 4 · 도구 목록을 주고 한 번 묻는다
schemas = [t.schema() for t in TOOLS]
r = llm.chat(msgs, tools=schemas, max_tokens=CFG["max_tokens"])
print("글 답 :", repr(r.content[:200]))
print("도구 호출 :", json.dumps(r.tool_calls, ensure_ascii=False, indent=1))
print("\n여기까지 우리는 아무 함수도 실행하지 않았다 — 모델은 호출을 생성했을 뿐이다.")

### 셀 5 · 호출을 우리가 실행해 되먹인다 — 손으로 한 바퀴
- 돌아온 호출의 `name` · `arguments` 로 **우리가** 함수를 실행하고, 결과를 `role: "tool"` 메시지로 컨텍스트에 붙인다
- 그다음 다시 모델을 부르면, 모델은 관찰을 읽고 **다음 행동**을 낸다. 이 한 바퀴를 자동으로 반복하는 것이 루프다

In [ ]:
# 셀 5 · 손으로 한 바퀴
msgs2 = list(msgs)
if r.tool_calls:
    c = r.tool_calls[0]
    msgs2.append({"role": "assistant", "content": r.content or "",
                  "tool_calls": [{"id": c["id"], "type": "function", "function": {"name": c["name"], "arguments": c["raw"]}}]})
    obs = next(t for t in TOOLS if t.name == c["name"]).fn(**c["arguments"])
    msgs2.append({"role": "tool", "tool_call_id": c["id"], "content": str(obs)})
    print("관찰(우리가 실행한 결과) :", str(obs)[:160])
    r2 = llm.chat(msgs2, tools=schemas, max_tokens=CFG["max_tokens"])
    print("다음 행동 :", [(x["name"], x["arguments"]) for x in r2.tool_calls] or ("최종 답 → " + r2.content[:200]))
else:
    print("이번에는 모델이 도구를 부르지 않았다 — 도구 호출이 약한 모델이면 이런 일이 생긴다(실습 04 에서 모델 크기별로 측정)")

### 셀 6 · 루프를 직접 짠다
강의의 의사코드를 그대로 옮긴다.

```
상태 = [system, user]
while 미완료 and 예산 남음:
    행동 = LLM(상태)            # tool_call 또는 최종답변
    if 최종답변: break
    관찰 = 실행(행동)           # 런타임이 도구 실행
    상태 += [행동, 관찰]        # 결과를 컨텍스트에 누적
```
- `for step in range(max_steps)` 가 「예산 남음」이고, `return` 이 「최종답변이면 끝」이다
- 이 함수 하나가 에이전트의 전부다

In [ ]:
# 셀 6 · 루프를 직접 짠다
def my_agent(question, max_steps=CFG["max_steps"]):
    by = {t.name: t for t in TOOLS}
    msgs = [{"role": "system", "content": SYSTEM_D}, {"role": "user", "content": question}]     # 상태 = 메시지 열
    for step in range(max_steps):                                                                # 예산 남음
        r = llm.chat(msgs, tools=schemas, max_tokens=CFG["max_tokens"])                          # 행동 = LLM(상태)
        if not r.tool_calls:                                                                     # 최종답변이면 끝
            msgs.append({"role": "assistant", "content": r.content})
            return r.content, msgs
        msgs.append({"role": "assistant", "content": r.content or "",
                     "tool_calls": [{"id": c["id"], "type": "function", "function": {"name": c["name"], "arguments": c["raw"]}} for c in r.tool_calls]})
        for c in r.tool_calls:
            obs = by[c["name"]].fn(**c["arguments"])                                             # 관찰 = 실행(행동) — 런타임이 한다
            msgs.append({"role": "tool", "tool_call_id": c["id"], "content": str(obs)})          # 상태 += [행동, 관찰]
    return None, msgs                                                                            # 예산을 다 썼다
print("루프 함수를 만들었다 :", my_agent.__name__)

### 셀 7 · 루프를 돌리고 메시지 열을 들여다본다
- 컨텍스트 = 메시지 열이다. 한 스텝마다 열이 길어진다
- 각 메시지의 `role` 을 본다 : `system`(지시) · `user`(요청) · `assistant`(모델 출력 — 도구 호출 포함) · `tool`(실행 결과)

In [ ]:
# 셀 7 · 루프를 돌리고 메시지 열을 본다
ans, hist = my_agent(Q["q"])
for i, m in enumerate(hist):
    what = m["content"] if m["content"] else ("tool_calls → " + ", ".join(f'{c["function"]["name"]}({c["function"]["arguments"]})' for c in m.get("tool_calls", [])))
    print(f"{i:>2} [{m['role']:<9}] {str(what).replace(chr(10), ' ')[:110]}")
print("\n최종 답 :", ans)
print("채점 :", score_D(Q, ans or ""))

### 셀 8 · 완성형 `run_agent` — 궤적 그림 · 컨텍스트 누적
- `agent_lab.run_agent` 는 셀 6 의 루프에 **기록과 안전장치**를 더한 것이다(없는 도구 · 깨진 인자 · 최대 스텝)
- 왼쪽 : 궤적(LLM → 도구 호출 → 관찰이 이어짐). 오른쪽 : 스텝마다 모델이 읽은 입력 토큰 — 컨텍스트가 쌓이는 만큼 **매 스텝 읽는 양이 커진다**

In [ ]:
# 셀 8 · 완성형 · 궤적 그림 · 컨텍스트 누적
traj = run_agent(llm, SYSTEM_D, Q["q"], TOOLS, max_steps=CFG["max_steps"], max_tokens=CFG["max_tokens"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, max(3.2, 0.5 * (2 + 3 * len(traj.steps)))), gridspec_kw={"width_ratios": [1.5, 1]})
draw_trajectory(traj, title="궤적", ax=a1)
draw_context_growth(traj, ax=a2, title="스텝마다 읽은 입력 토큰")
plt.tight_layout(); plt.show()
print("호출한 도구 :", traj.calls, "| 끝난 이유 :", traj.stopped, "| 통계 :", traj.trace.stats())
print("채점 :", score_D(Q, traj.final))

### 셀 9 · 루프의 안전장치
- 모델이 **없는 도구**를 부르면 : 실행할 수 없다 → 오류를 관찰로 돌려줘 모델이 고칠 기회를 준다
- 모델이 **끝나지 않으면** : 루프가 최대 스텝에서 끊는다. 「끝낼지」는 모델이 정하지만 「더는 못 돈다」는 코드가 정한다
- 이 셀은 진짜 모델이 아니라 **가짜 정책**(`ScriptedPolicy`)으로 두 상황을 일부러 만든다. 서버가 없어도 돌고, 같은 결과가 나온다

In [ ]:
# 셀 9 · 안전장치 — 가짜 정책으로 두 상황을 일부러 만든다
steps_ok, final_ok = plan_D(Q, orders, cust)
def stumble(messages, tools, rng):            # 첫 호출은 없는 도구, 그다음부터 정답 경로
    n = sum(1 for m in messages if m["role"] == "assistant" and m.get("tool_calls"))
    if n == 0:
        return ("call", "delete_all_orders", {})
    k = n - 1
    return ("call", *steps_ok[k]) if k < len(steps_ok) else ("final", final_ok)
def forever(messages, tools, rng):            # 끝내지 않고 같은 도구만 계속 부름
    return ("call", "get_policy", {})
t1 = run_agent(ScriptedPolicy(stumble), SYSTEM_D, Q["q"], TOOLS, max_steps=8)
t2 = run_agent(ScriptedPolicy(forever), SYSTEM_D, Q["q"], TOOLS, max_steps=4)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))
draw_trajectory(t1, title="없는 도구를 부른 뒤 스스로 고침(끝난 이유: %s)" % t1.stopped, ax=a1)
draw_trajectory(t2, title="끝나지 않는 정책 → 최대 스텝에서 끊김(끝난 이유: %s)" % t2.stopped, ax=a2, show_obs=False)
plt.tight_layout(); plt.show()

### 셀 10 · 단발 vs 루프 — 같은 질문 8개를 측정한다
- 같은 모델 · 같은 8개 질문을 **도구 없이 한 번**(단발)과 **도구 + 루프**로 각각 푼다
- 서로 독립인 실행은 `pmap` 으로 겹쳐 돈다(동시에 몇 개인지는 서버 설정에 달림). 지연 값은 겹쳐 돌 때 단독 실행보다 길게 나올 수 있다
- 성공은 정답 금액 · 판정이 답에 들어 있는지 규칙으로 본다. 토큰 · 지연은 호출 전체의 합이다(캐시된 응답은 처음 호출했을 때 측정한 값)
- 루프가 더 정확한 대신 더 비싸다는 것이 숫자로 나오는지 본다. **기대와 달라도 그대로 남긴다**

In [ ]:
# 셀 10 · 단발 vs 루프 — 질문 8개
def one(q):
    m0 = [{"role": "system", "content": SYSTEM_D}, {"role": "user", "content": q["q"]}]
    r0 = llm.chat(m0, max_tokens=CFG["max_tokens"])
    tr = run_agent(llm, SYSTEM_D, q["q"], TOOLS, max_steps=CFG["max_steps"], max_tokens=CFG["max_tokens"])
    return {"질문": q["oid"], "단발 성공": score_D(q, r0.content)["성공"], "루프 성공": score_D(q, tr.final)["성공"], "단발 토큰": r0.tokens, "루프 토큰": tr.trace.tokens,
            "루프 호출": tr.trace.llm_calls, "단발 지연": r0.latency, "루프 지연": tr.trace.latency, "끝난 이유": tr.stopped}
rows = pmap(one, QS1)          # 질문 8개는 서로 독립 → 겹쳐 돌린다(결과는 질문 순서 그대로)
df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
S = {"단발 성공률": df["단발 성공"].mean(), "루프 성공률": df["루프 성공"].mean(), "단발 평균 토큰": df["단발 토큰"].mean(), "루프 평균 토큰": df["루프 토큰"].mean(),
     "루프 평균 LLM 호출": df["루프 호출"].mean(), "단발 평균 지연": df["단발 지연"].mean(), "루프 평균 지연": df["루프 지연"].mean()}
print({k: round(float(v), 2) for k, v in S.items()})
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
bars(ax[0], ["단발", "루프"], [S["단발 성공률"], S["루프 성공률"]], colors=[COLOR["io"], COLOR["llm"]], title="성공률(질문 8개)", fmt="{:.2f}")
bars(ax[1], ["단발", "루프"], [S["단발 평균 토큰"], S["루프 평균 토큰"]], colors=[COLOR["io"], COLOR["llm"]], title="질문당 토큰", fmt="{:.0f}")
bars(ax[2], ["단발", "루프"], [S["단발 평균 지연"], S["루프 평균 지연"]], colors=[COLOR["io"], COLOR["llm"]], title="질문당 지연(초)", fmt="{:.1f}")
plt.tight_layout(); plt.show()

### 셀 11 · 정리 · 결과 저장

In [ ]:
# 셀 11 · 정리 · 결과 저장
RES = {"설정": {"모델": llm.model, **CFG, "질문 수": len(QS1)}, "단발 vs 루프(질문 8개)": {k: float(v) for k, v in S.items()},
       "한 질문의 호출 순서": traj.calls, "한 질문의 스텝 수": traj.n_steps, "안전장치": {"없는 도구 뒤 끝난 이유": t1.stopped, "끝나지 않는 정책의 끝난 이유": t2.stopped}}
save_measured("01", RES)
print("에이전트 = 모델 + 도구 + 루프. 모델이 한 것은 매 스텝 「다음 행동」을 글로 내는 것뿐이고, 실행 · 누적 · 종료는 루프(코드)가 했다.")

### 확인 질문
1. 셀 4 에서 돌아온 `도구 호출`은 누가 실행했는가? 「호출을 생성한다」와 「호출을 실행한다」가 다른 이유를 `arguments` 가 문자열인 점, 그리고 셀 5 에서 우리가 직접 한 일과 연결해 설명해 보자.
2. 셀 10 에서 루프는 단발보다 토큰을 몇 배 썼는가? 그 증가는 스텝 수에 비례하는가, 스텝이 늘수록 더 가파른가? 셀 8 의 컨텍스트 누적 그림으로 설명해 보자.
3. 셀 9 의 두 번째 정책처럼 끝나지 않는 경우를 막는 장치는 코드 어디에 있었는가? 「끝낼지」는 모델이 정하는데 「더는 못 돈다」는 모델이 아니라 루프에 두어야 하는 이유는 무엇인가?